# 00 — Validate Instanton and IC Generation

Pure-Python validation — no C++ binaries needed.

Compares `ptbuilder` outputs against the reference values from
`reference_codes/notebooks/Simulation_Planner-N2_gamma2.ipynb`
for the canonical case: `lambda_bar=0.84`, `gamma≈2`.

Each section prints a side-by-side comparison and flags discrepancies.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import ptbuilder as pt
from scipy import optimize
from pathlib import Path

lambda_bar = 0.84

def check(label, ours, ref, tol=1e-4):
    ok = abs(ours - ref) <= tol * max(abs(ref), 1e-12)
    status = 'OK  ' if ok else 'FAIL'
    print(f"  [{status}]  {label:<30s}  ours={ours:.6f}   ref={ref:.6f}")
    return ok

## 1. Potential coefficients

The reference notebook (`Simulation_Planner`) computes `m2`, `delta`, `lambda`
inline for `lambda_bar=0.84`.  Our `Phi4Potential` should give identical values.

In [ ]:
# Reference: Simulation_Planner-N2_gamma2.ipynb, cell 4
Upsilon_ref = 3.0 + np.sqrt(9.0 - 8.0 * lambda_bar)
m2_ref      = 0.5
delta_ref   = -(3.0 + np.sqrt(9.0 - 8.0 * lambda_bar)) / (4.0 * lambda_bar)
lam_ref     = (3.0 + np.sqrt(9.0 - 8.0 * lambda_bar))**2 / (32.0 * lambda_bar)

pot = pt.Phi4Potential(lambda_bar)

print("Potential coefficients (lambda_bar=0.84):")
check("m2",    pot.m2,    m2_ref)
check("delta", pot.delta, delta_ref)
check("lam",   pot.lam,   lam_ref)
print(f"\n  phi_false = {pot.phi_false},  phi_true = {pot.phi_true}")

## 2. Instanton profile

Both the reference and our code call `cosmoTransitions.SingleFieldInstanton`
with the same arguments, so the profiles should be identical.

We compare `rin_0`, `rmid_0`, `rout_0` by running both approaches here.

In [ ]:
from cosmoTransitions.tunneling1D import SingleFieldInstanton

# Reference approach (from Simulation_Planner)
raw = SingleFieldInstanton(1.0, 0.0, pot.V, pot.dV, alpha=3).findProfile(
    xtol=1e-5, phitol=1e-5
)
phi0 = raw.Phi[0]
phi_in_ref  = phi0 * (1 - np.tanh(-0.5)) / 2
phi_out_ref = phi0 * (1 - np.tanh( 0.5)) / 2
phi_mid_ref = phi0 * 0.5

rin_0_ref  = optimize.brentq(lambda r: np.interp(r, raw.R, raw.Phi) - phi_in_ref,  raw.R[0], raw.R[-1])
rmid_0_ref = optimize.brentq(lambda r: np.interp(r, raw.R, raw.Phi) - phi_mid_ref, raw.R[0], raw.R[-1])
rout_0_ref = optimize.brentq(lambda r: np.interp(r, raw.R, raw.Phi) - phi_out_ref, raw.R[0], raw.R[-1])

# Our approach
config  = pt.Config()
model   = pt.PhysicsModel(pt.Phi4Potential(lambda_bar), config)
profile = model.instanton

print("Instanton radii:")
check("rin_0",  profile.rin_0,  rin_0_ref)
check("rmid_0", profile.rmid_0, rmid_0_ref)
check("rout_0", profile.rout_0, rout_0_ref)

plt.figure(figsize=(7, 4))
plt.plot(raw.R, raw.Phi, 'k-', label='reference')
plt.plot(profile.R, profile.Phi, 'r--', lw=1, label='ours')
plt.axvline(profile.rmid_0, ls=':', color='C2', label=f'rmid_0={profile.rmid_0:.3f}')
plt.xlabel('r');  plt.ylabel(r'$\phi$')
plt.title('Instanton profile')
plt.legend();  plt.tight_layout()

## 3. Bubble kinematics — gamma=2 case

The reference uses `d = 27.703` (bubble separation) and verifies it gives `gamma ≈ 2`
at the collision time.  We reproduce that here with our `BubbleKinematics`.

In [ ]:
d_ref = 27.703   # from Simulation_Planner

kin = model.kinematics

# Reference: find t* where Rc(t*) = d/2
t_coll_ref = optimize.brentq(lambda t: kin.R(t, r='mid') - d_ref/2, 0, 1e4)
gamma_ref  = float(kin.Gamma(t_coll_ref))

# Our collision_time() method
t_coll_ours = kin.collision_time(d_ref, 0., 0.)
gamma_ours  = float(kin.Gamma(t_coll_ours))

# Implied gamma from d = 2 * gamma * rmid_0
gamma_from_d = d_ref / (2.0 * profile.rmid_0)

print("Gamma=2 validation (d=27.703):")
print(f"  Reference brentq:   t_coll={t_coll_ref:.4f},  gamma={gamma_ref:.5f}")
print(f"  Our collision_time: t_coll={t_coll_ours:.4f},  gamma={gamma_ours:.5f}")
print(f"  d / (2*rmid_0):     gamma={gamma_from_d:.5f}  (IC convention)")
print()
check("gamma at collision", gamma_ours, gamma_ref)

# What d gives exactly gamma=2.0 with our rmid_0?
d_gamma2 = 2.0 * 2.0 * profile.rmid_0
print(f"\n  d for exact gamma=2.0 via our IC: {d_gamma2:.4f}  (ref used {d_ref})")

## 4. Cutoff parameters

The reference notebook derives cutoff parameters as:
```
smax  = 1.2 * d
t_cut = 0.9 * smax  = 1.08 * d
t_0   = 0.25 * (smax - t_cut)  = 0.03 * d
t_max = t_cut + 7 * t_0
```
Our `_cutoff_times()` in `ic.py` uses different coefficients — flagged below.

In [ ]:
from ptbuilder.ic import _cutoff_times

d = d_ref

# Reference formula (Simulation_Planner)
smax_ref  = 1.2 * d
tc_ref    = 0.9 * smax_ref
t0_ref    = 0.25 * (smax_ref - tc_ref)
tmax_ref  = tc_ref + 7.0 * t0_ref

# Our formula
t0_ours, tc_ours, tm_ours, tmax_ours = _cutoff_times(d, cutoff_type=0, t_0_scal=1.0)
smax_ours = max(1.2 * d, 1.2 * tmax_ours)  # from write_2d_setup

print("Cutoff parameters for d=27.703 (cutoff_type=0):")
print(f"  {'parameter':<10}  {'ours':>12}  {'reference':>12}  {'ratio':>8}")
print(f"  {'-'*48}")
for name, o, r in [
    ('t_cut',  tc_ours,   tc_ref),
    ('t_0',    t0_ours,   t0_ref),
    ('t_max',  tmax_ours, tmax_ref),
    ('smax',   smax_ours, smax_ref),
]:
    ratio = o / r
    flag  = '' if abs(ratio - 1) < 0.01 else '  <-- DIFFERS'
    print(f"  {name:<10}  {o:>12.4f}  {r:>12.4f}  {ratio:>8.3f}{flag}")

## 5. IC HDF5 sanity check

Write a setup file for `gamma=2` and verify the contents look physically reasonable.

In [ ]:
import tempfile, os

params = pt.BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                                how_often_ds=5, baby_steps=100,
                                cutoff_type=0)
times  = np.linspace(1., 63., 16)

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'setup_gamma2.h5'
    pt.write_2d_setup(model, gamma=2.0, times=times, path=path, params=params)

    with h5py.File(path, 'r') as f:
        attrs = dict(f.attrs)
        z     = f['z'][:]
        phi0  = f['phi0'][:]
        wlist = f['wlist'][:]

print("HDF5 attributes written:")
for k, v in sorted(attrs.items()):
    print(f"  {k:<16} = {v}")

print(f"\nz grid:   n={len(z)},  range=[{z[0]:.3f}, {z[-1]:.3f}]")
print(f"phi0:     min={phi0.min():.4f},  max={phi0.max():.4f}")
print(f"wlist:    n={len(wlist)},  range=[{wlist[0]:.4f}, {wlist[-1]:.4f}]")

print("\nSanity checks:")
print(f"  [{'OK' if phi0.min() >= -0.1 else 'FAIL'}]  phi0 >= 0 (vacuum = 0)")
print(f"  [{'OK' if phi0.max() >  0.5  else 'FAIL'}]  phi0 reaches true vacuum")
print(f"  [{'OK' if attrs['t_cut'] < attrs['smax'] else 'FAIL'}]  t_cut < smax")
print(f"  [{'OK' if attrs['ds']   > 0  else 'FAIL'}]  ds > 0")

## 6. Initial field profile

Plot `phi0(z)` from the IC file and verify it looks like two back-to-back bubble walls.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'setup_gamma2.h5'
    pt.write_2d_setup(model, gamma=2.0, times=times, path=path, params=params)
    with h5py.File(path, 'r') as f:
        z    = f['z'][:]
        phi0 = f['phi0'][:]
        d    = float(f.attrs['d'])

plt.figure(figsize=(9, 4))
plt.plot(z, phi0, 'k-')
plt.axvline(d/2, ls='--', color='C0', label=f'collision centre (z={d/2:.2f})')
plt.xlabel('z')
plt.ylabel(r'$\phi_0$')
plt.title(f'Initial field profile  (gamma=2,  d={d:.3f})')
plt.legend()
plt.tight_layout()